COSC 301 Project
Group 4
Instacart Market Basket Analysis

**Importing Files**

The way to run this program is by running bash scripts/get_data.sh in terminal 

In [34]:
import pandas as pd               # for data manipulation
import matplotlib.pyplot as plt   # for plotting 
import seaborn as sns             # for statistical graph
import sqlite3                    # for connecting to the database
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import numpy as np

In [35]:
orders = pd.read_csv('../raw_data/orders.csv' )
products = pd.read_csv('../raw_data/products.csv')
order_products_prior = pd.read_csv('../raw_data/order_products__prior.csv')
aisles = pd.read_csv('../raw_data/aisles.csv')
order_products_train = pd.read_csv('../raw_data/order_products__train.csv')

In [36]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


## Research Question 1

How accurately will we recommend the top 5 products a user is most likely to purchase in their next order based on their previous orders (using the order_product_prior csv file)?

Steps taken:

Step 1: Create dataframe with all orders of each user along with details of the products in the order.

Step 2: From per-user, per-product history, calculate the number of times a user bought a product, last order they bought it in, and the reorder rate for that product for that user.

Step 3: Implement the 2 recommendation algorithms: the first one recommends top 5 products based on frequency of ordering the product that the user has order in all of their order. The second one uses the popular products in each aisle and the popular aisles that the user orders from and recommends those top 5 items.

Step 4: Run the 2 recommendation algorithms. Compute metrics by comparing the recommended 5 products with the training set of products for the users.

In [37]:
# Join prior order-products with orders to get user_id + order_number
op_prior = order_products_prior.merge(
    orders[['order_id', 'user_id', 'order_number']],
    on='order_id',
    how='left'
)

# add product meta (name/aisle/department)
prod_meta = products[['product_id', 'product_name', 'aisle_id', 'department_id']].copy()
op_prior = op_prior.merge(prod_meta, on='product_id', how='left')

# now we have a dataframe with info order, products in the order, and user who made the order.
op_prior.head()


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,product_name,aisle_id,department_id
0,2,33120,1,1,202279,3,Organic Egg Whites,86,16
1,2,28985,2,1,202279,3,Michigan Organic Kale,83,4
2,2,9327,3,0,202279,3,Garlic Powder,104,13
3,2,45918,4,1,202279,3,Coconut Butter,19,13
4,2,30035,5,0,202279,3,Natural Sweetener,17,13


Here, we create the users' ordered products statistics and then the 1st recommendation algorithm which recommends the top 5 most frequently ordered products for the specified user.

In [38]:
# Per-user, per-product stats from PRIOR history
# calculating the numbers of times a user bought a product, the last order number they bought it in, 
# and the reorder rate for that product for that user.
user_prod_stats = (
    op_prior.groupby(['user_id', 'product_id'], as_index=False)
    .agg(
        times_bought=('order_id', 'count'),
        last_order_number=('order_number', 'max'),
        reorder_rate=('reordered', 'mean')
    )
)

# merge names for readability
user_prod_stats = user_prod_stats.merge(prod_meta, on='product_id', how='left')

# recommendation function based on frequency of a product being ordered; accepts user id and number of recommendations to make
def recommend_top5_frequency(user_id: int, n: int = 5):
    """
    - prefer items bought often
    - prefer items bought recently
    - prefer items with high reorder_rate
    """
    
    df = user_prod_stats[user_prod_stats['user_id'] == user_id].copy()
    if df.empty:
        return pd.DataFrame(columns=['product_id','product_name','times_bought','reorder_rate','last_order_number'])

    df = df.sort_values(
        by=['times_bought', 'last_order_number', 'reorder_rate'],
        ascending=[False, False, False]
    )
    return df[['user_id','product_id','product_name','times_bought','reorder_rate','last_order_number']].head(n)

# try
recommend_top5_frequency(1)

,user_id,product_id,product_name,times_bought,reorder_rate,last_order_number
0,1,196,Soda,10,0.900000,10
3,1,12427,Original Beef Jerky,10,0.900000,10
1,1,10258,Pistachios,9,0.888889,10
8,1,25133,Organic String Cheese,8,0.875000,10
4,1,13032,Cinnamon Toast Crunch,3,0.666667,10


Now we calculate the global product popularity, users' preferred aisles, and the aisles' popular products. After this we create the 2nd recommendation algorithm which recommended the top 5 products a user orders from the aisles they most popularly order from.

In [39]:
# Global popularity + reorder rate per product (from op_prior)
# aggregate variables are global_times and global_reorder_rate
global_prod_stats = (
    op_prior.groupby('product_id', as_index=False)
    .agg(
        global_times=('order_id', 'count'),
        global_reorder_rate=('reordered', 'mean')
    )
).merge(prod_meta, on='product_id', how='left')

# User aisle preferences
#aisle_count is the number of a times the aisle was mentioned in a prior order for a user.
user_aisle_pref = (
    op_prior.groupby(['user_id', 'aisle_id'], as_index=False)
    .agg(aisle_count=('order_id', 'count'))
)

# Popular products within each aisle
#this is different from user_aisle_pref because it is number of times a product in an aisle was ordered in general and not specific to a user.
aisle_prod_pop = (
    op_prior.groupby(['aisle_id', 'product_id'], as_index=False)
    .agg(aisle_prod_count=('order_id', 'count'))
).merge(global_prod_stats, on=['product_id','aisle_id'], how='left')


def recommend_top5_aisle(user_id: int, n: int = 5, top_aisles: int = 3):
    """
    Content-based recommender:
    1) find user's top aisles
    2) recommend popular products from those aisles
       (rank by aisle popularity + global reorder rate)
    """
    
    # top aisles for the specified user
    top_aisles_for_user = (
        user_aisle_pref[user_aisle_pref['user_id'] == user_id]
        .sort_values('aisle_count', ascending=False)
        .head(top_aisles)['aisle_id']
        .tolist()
    )
        
    if not top_aisles_for_user:
        return pd.DataFrame(columns=['product_id','product_name','aisle_id','aisle_prod_count','global_reorder_rate'])

    popular_prod_AND_in_top_aisle_for_user = aisle_prod_pop[aisle_prod_pop['aisle_id'].isin(top_aisles_for_user)].copy()
    
    # recommend products which could be new or a reorder for the user.
    # sorting products to recommend by aisle
    popular_prod_AND_in_top_aisle_for_user = popular_prod_AND_in_top_aisle_for_user.sort_values(
        by=['aisle_prod_count', 'global_reorder_rate', 'global_times'],
        ascending=[False, False, False]
    )

    return popular_prod_AND_in_top_aisle_for_user[['product_id','product_name','aisle_id','aisle_prod_count','global_reorder_rate']].head(n)

# try
recommend_top5_aisle(1)

,product_id,product_name,aisle_id,aisle_prod_count,global_reorder_rate
26966,196,Soda,77,35791,0.776480
27077,10957,Fridge Pack Cola,77,18269,0.715255
8257,3599,Baked Aged White Cheddar Rice and Corn Puffs,23,13691,0.643635
8337,15902,100 Calorie Per Bag Popcorn,23,12822,0.678209
44079,35140,Organic Whole Cashews,117,12816,0.605025


Below we create the truth set which is based on the train dataset. After that, we evaluate the 2 recommendation algorithms using precision and recall metrics.

In [40]:
# Build user -> set of products in their TRAIN order (the "next order" we are trying to predict)
op_train = order_products_train.merge(
    orders[['order_id', 'user_id']],
    on='order_id',
    how='left'
)

truth = (
    op_train.groupby('user_id')['product_id']
    .apply(lambda s: set(s.values))
    .to_dict()
)

import random

def precision_at_k(recs, truth_set, k=5):
    recs = list(recs)[:k]
    if not recs:
        return 0.0
    hits = len(set(recs) & truth_set)
    return hits / k

def recall_at_k(recs, truth_set, k=5):
    recs = list(recs)[:k]
    if not truth_set:
        return 0.0
    hits = len(set(recs) & truth_set)
    return hits / len(truth_set)

def eval_recommender(user_ids, recommender_fn, k=5):
    ps, rs = [], []
    for uid in user_ids:
        if uid not in truth:
            continue
        rec_df = recommender_fn(uid, n=k)
        recs = rec_df['product_id'].tolist() if 'product_id' in rec_df else []
        ps.append(precision_at_k(recs, truth[uid], k))
        rs.append(recall_at_k(recs, truth[uid], k))
    return (sum(ps)/len(ps), sum(rs)/len(rs)) if ps else (0.0, 0.0)

# sample users for speed
all_users = list(truth.keys())
sample_users = random.sample(all_users, k=min(2000, len(all_users)))

p1, r1 = eval_recommender(sample_users, recommend_top5_frequency, k=5)
p2, r2 = eval_recommender(sample_users, recommend_top5_aisle, k=5)

print("Baseline Frequency  P@5:", round(p1,4), " R@5:", round(r1,4))
print("Aisle Content-Based P@5:", round(p2,4), " R@5:", round(r2,4))

Baseline Frequency  P@5: 0.3612  R@5: 0.2265
Aisle Content-Based P@5: 0.0977  R@5: 0.0514


Based on the above metrics printed, the performance is much better for the frequency based method rather than the aisle popularity method of recommendation.

Now let us see some other algorithms and how the performance is when using them to make the recommendations.

### Item based collaborative filtering

The logic in this method is that if a user buys an item, another item similar to this one would also be bought.

In [41]:
# Taking the top 5000 popularly ordered products to for the CF model to run well on my machine.
top_products = op_prior.groupby('product_id')['order_id'].count().nlargest(5000).index

# Filtering the prior orders which have a top product and also a user in the sample_users list
op_prior_filtered = op_prior[
    op_prior['product_id'].isin(top_products) &
    op_prior['user_id'].isin(sample_users)
]

# Build user-product matrix which is a pivot table with the count of orders for each user-product pair.
user_product_matrix = op_prior_filtered.pivot_table(
    index='user_id',
    columns='product_id',
    values='order_id',
    aggfunc='count',
    fill_value=0
)

# Convert to sparse matrix for efficiency
sparse_matrix = csr_matrix(user_product_matrix.values)

Here we are calculating the item similarity between all the items in the sparse imatrix

In [42]:
# Cosine similarity between products. taking the transpose so that products are in the rows
item_similarity = cosine_similarity(sparse_matrix.T)

# Wrapping in a DataFrame for easy lookup
item_sim_df = pd.DataFrame(
    item_similarity,
    index=user_product_matrix.columns,
    columns=user_product_matrix.columns
)

Below is the recommendation algorithm based on Item-based Collaborative filtering

In [43]:
def recommend_item_based_cf(user_id, n=5, n_similar=10):
    if user_id not in user_product_matrix.index:
        return pd.DataFrame(columns=['product_id'])
    
    # products this user has already bought before.
    user_purchases = user_product_matrix.loc[user_id]
    bought_products = user_purchases[user_purchases > 0].index.tolist()
    
    if not bought_products:
        return pd.DataFrame(columns=['product_id'])
    
    # Accumulate similarity scores across all bought products
    scores = pd.Series(dtype=float)
    for product in bought_products:
        if product not in item_sim_df.index:
            continue
        # get top N similar products to the current one
        similar = item_sim_df[product].drop(index=bought_products, errors='ignore')
        similar = similar.nlargest(n_similar)
        scores = scores.add(similar, fill_value=0)
    
    # sort by the score and take top n products to recommend
    top_products = (
        scores.sort_values(ascending=False)
        .head(n)
        .reset_index()
        .rename(columns={'index': 'product_id', 0: 'score'})
    )
    return top_products[['product_id']]

Now, we will evaluate the recommendation algorithm against the truth set.

In [44]:
# evaluate the CF recommendation algorithm on users present in the user-product matrix and in the truth set
cf_users = set(user_product_matrix.index)
sample_users_cf = [u for u in sample_users if u in cf_users and u in truth]

print(f"Eligible users for CF eval: {len(sample_users_cf)}")

p3, r3 = eval_recommender(sample_users_cf, recommend_item_based_cf, k=5)
print("Item-Based CF P@5:", round(p3, 4), " R@5:", round(r3, 4))

Eligible users for CF eval: 1999
Item-Based CF P@5: 0.0064  R@5: 0.0024
